In [1]:
# 03_latency_overhead.ipynb
#
# Experiment 3 -- Per-request latency of the governed decision path.
#
# Purpose:
#   Measure the added per-request latency of the governed path (model inference
#   plus a group-wise decision and an append-only audit-record write) relative to
#   a plain model call, over many requests. This bounds the real-time cost of the
#   mitigation. The measurement is of the instantiated path only; a production
#   deployment adding cryptographic audit chaining or synchronous explanations
#   would incur more.
#
# Outputs (PNG+PDF, 600 dpi, grayscale, no captions; paths not printed):
#   results/tables/exp3_latency.csv
#   results/figures/exp3_latency.(png|pdf)

import os, time, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings("ignore")
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if not os.path.isdir(os.path.join(PROJECT_ROOT, "results")):
    PROJECT_ROOT = os.getcwd()
FIG_DIR = os.path.join(PROJECT_ROOT, "results", "figures")
TAB_DIR = os.path.join(PROJECT_ROOT, "results", "tables")
for _d in (FIG_DIR, TAB_DIR):
    os.makedirs(_d, exist_ok=True)

sns.set_theme(style="whitegrid")
GRAYS = ["#000000", "#555555", "#999999", "#cccccc"]
sns.set_palette(sns.color_palette(GRAYS))
plt.rcParams.update({"figure.dpi": 600, "savefig.dpi": 600, "font.size": 11,
    "axes.edgecolor": "black", "axes.linewidth": 0.8, "grid.color": "0.85"})

def save_fig(fig, name):
    fig.savefig(os.path.join(FIG_DIR, name + ".png"), dpi=600, bbox_inches="tight")
    fig.savefig(os.path.join(FIG_DIR, name + ".pdf"), dpi=600, bbox_inches="tight")

# Train a small model on synthetic-like data
rng = np.random.default_rng(0)
N = 20000
X = rng.normal(0, 1, (N, 8)); g = rng.binomial(1, 0.4, N)
y = (X[:,0] - X[:,1] + rng.normal(0,0.5,N) > 0).astype(int)
sc = StandardScaler().fit(X); m = LogisticRegression(max_iter=1000).fit(sc.transform(X), y)
t1 = np.quantile(m.predict_proba(sc.transform(X))[:,1][g==1], 0.5)
t0 = np.quantile(m.predict_proba(sc.transform(X))[:,1][g==0], 0.5)

AUDIT = []
def baseline_call(x):
    p = m.predict_proba(sc.transform(x.reshape(1,-1)))[:,1][0]
    return int(p >= 0.5)
def governed_call(x, grp):
    p = m.predict_proba(sc.transform(x.reshape(1,-1)))[:,1][0]
    thr = t1 if grp == 1 else t0
    decision = int(p >= thr)
    AUDIT.append((float(p), int(grp), decision))   # append-only audit write
    return decision

REQUESTS = 5000
xs = rng.normal(0, 1, (REQUESTS, 8)); gs = rng.binomial(1, 0.4, REQUESTS)
# warmup
for i in range(200): baseline_call(xs[i]); governed_call(xs[i], gs[i])
AUDIT.clear()

t = time.perf_counter()
for i in range(REQUESTS): baseline_call(xs[i])
base_ms = (time.perf_counter()-t)/REQUESTS*1000

t = time.perf_counter()
for i in range(REQUESTS): governed_call(xs[i], gs[i])
gov_ms = (time.perf_counter()-t)/REQUESTS*1000

overhead = (gov_ms-base_ms)/base_ms*100
res = pd.DataFrame([{"pipeline":"baseline_model","latency_ms":round(base_ms,4),"overhead_pct":np.nan},
                    {"pipeline":"governed_wrapper","latency_ms":round(gov_ms,4),"overhead_pct":round(overhead,1)}])
res.to_csv(os.path.join(TAB_DIR,"exp3_latency.csv"),index=False)
print(res.to_string(index=False))
print(f"Absolute added latency per request: {gov_ms-base_ms:.4f} ms")

fig,ax=plt.subplots(figsize=(5,4))
ax.bar([0,1],[base_ms,gov_ms],edgecolor="black",linewidth=1.0,color=[GRAYS[0],GRAYS[2]])
ax.set_xticks([0,1]); ax.set_xticklabels(["baseline","governed"])
ax.set_ylabel("Latency (ms / request)")
fig.tight_layout(); save_fig(fig,"exp3_latency"); plt.close(fig)
print("Done: latency overhead measurement.")


        pipeline  latency_ms  overhead_pct
  baseline_model      0.1407           NaN
governed_wrapper      0.1448           2.9
Absolute added latency per request: 0.0041 ms
Done: latency overhead measurement.
